# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [ ]:
%load_ext dotenv
%dotenv ../05_src/.secrets # Load environment variables from the .secrets file

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "C:\\Users\\james\\Documents\\DSI\\deploying-ai\\02_activities\\documents\\ai_report_2025.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

document_text = ""
for page in docs: 
    document_text += page.page_content + "\n"

print(len(document_text))

53851


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [ ]:
from openai import OpenAI
from pydantic import BaseModel
import numpy as np
import os

# Initialize the OpenAI client with API Gateway URL and API key
client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                api_key='any value',
                default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})

In [ ]:
# Add dev prompt
system_prompt = "You speak in 19th century Victorian English."

In [ ]:
# Create an enhanced user prompt with specific instructions and formatting
user_prompt = """
    Given the following context from an article, do the following:
    
    1. Identify the article's title and author.
    2. Determine relevance by producing a statement, no longer than one paragraph, 
        that explains why is this article relevant for an AI professional in their professional development.
    3. Summarize concisely in no more than 1000 tokens while maintaining the prompted tone of {tone}.
    4. Describe the tone used to produce the summary.
    5. Give the number of input tokens and output tokens used in the response.
        
    The article is the following: 
    <article>
    {document_text}
    </article>

    Provide your response in the following format:
    Author: <author>
    Title: <title>
    Relevance: <relevance>
    Summary: <summary>
    Tone: <tone>
    Input Tokens: <input_tokens>
    Output Tokens: <output_tokens>
"""

In [ ]:
# Define a Pydantic model to structure the response from the model
class Format(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    Input_Tokens: int
    Output_Tokens: int

# Make a call to the OpenAI API using the client to generate the response according to the defined format
response = client.responses.parse(
    model="gpt-4o",
    instructions = system_prompt,
    input = [
        {"role": "developer", "content": system_prompt},
        {"role": "user", "content": user_prompt.format(document_text=document_text, tone="Victorian English")},
    ],
    text_format = Format,
)


In [ ]:
# Screen the screen to ensure correct function
response.output_parsed

Format(Author='Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari', Title='The GenAI Divide: State of AI in Business 2025', Relevance="This article is pertinent for AI professionals as it explores the challenges and strategies in implementing generative AI in enterprises. The insights on the 'GenAI Divide' highlight critical areas for improvement, offering strategies that can drive more effective AI adoption and integration.", Summary='In the treatise "The GenAI Divide: State of AI in Business 2025," penned by Aditya Challapally and colleagues, the authors explore the burgeoning gap in generative AI adoption within enterprises. Despite hefty investments, a mere five percent of organizations reap substantial benefits from AI initiatives, a disparity termed the \'GenAI Divide.\' The article elucidates that the core challenge lies not in model quality, regulations, or infrastructure; rather, it is the learning capabilities of these systems that falter. Noteworthy observations

In [ ]:
from IPython.display import display, Markdown

# Display the output text in Markdown format for better readability
display(Markdown(response.output_text))

{"Author":"Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari","Title":"The GenAI Divide: State of AI in Business 2025","Relevance":"This article is pertinent for AI professionals as it explores the challenges and strategies in implementing generative AI in enterprises. The insights on the 'GenAI Divide' highlight critical areas for improvement, offering strategies that can drive more effective AI adoption and integration.","Summary":"In the treatise \"The GenAI Divide: State of AI in Business 2025,\" penned by Aditya Challapally and colleagues, the authors explore the burgeoning gap in generative AI adoption within enterprises. Despite hefty investments, a mere five percent of organizations reap substantial benefits from AI initiatives, a disparity termed the 'GenAI Divide.' The article elucidates that the core challenge lies not in model quality, regulations, or infrastructure; rather, it is the learning capabilities of these systems that falter. Noteworthy observations include the widespread yet ineffective adoption of common tools and the shadow AI practices wherein employees employ personal AI tools for augmentation, indicating latent potential untapped by formal channels. The authors propose that successful organizations reduce this divide by eschewing traditional building for buying adaptive AI systems, thereby empowering front-line decision-making and prioritizing workflow integration over opulent features. Additionally, the piece foreshadows a future where interlinked agentic webs replace current monolithic systems, further emphasizing the need for adaptive technologies. Thus, this work serves as a beacon for AI professionals endeavoring to traverse the existing AI chasm and embrace an enhanced, adaptive, and integrative future.","Tone":"Analytical and insightful with a focus on future preparedness.","Input_Tokens":9966,"Output_Tokens":261}

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [71]:
from deepeval import evaluate
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase
from deepeval.models import GPTModel
from deepeval.metrics import SummarizationMetric
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

# Initialize the evaluation model with API Gateway URL and API key
model = GPTModel(
    model="gpt-4o-mini",
    temperature=0,
    #api_key='any_value',
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
)

# Define the summarization metric with specific assessment questions to evaluate the model's output
summarization = SummarizationMetric(
    threshold=0.5,
    model=model,
    assessment_questions=[
        "Does the summary capture the main points of the article accurately?",
        "Is the summary concise and free of unnecessary details?",
        "Does the summary conclude with the main point(s) of the article?",
        "Are the author and title correctly identified?",
        "Does the output use terms included in the article correctly and in context?"
    ]
)

# Define the coherence metric with specific evaluation steps to assess the clarity and logical flow of the model's output
coherence = GEval(
    name="Coherence",
    model=model,
    evaluation_steps=[
        "Does the response uses clear and direct language?",
        "Does the explanation avoid jargon or explains it when used?",
        "Are complex ideas presented in a way that's easy to follow?",
        "Are there any vague or confusing parts that reduce understanding?",
        "Does the output flow logically from one point to the next?"
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
)

# Define the tonality metric with specific evaluation steps to assess the appropriateness of the model's tone in the output
tonality = GEval(
    name="Tonality",
    model=model,
    evaluation_steps=[
        "Does the actual output maintains a consistent tone of {tone} throughout?",
        "Does the language in the actual output reflects expertise and {tone}-appropriate formality?",
        "Does the actual output stays contextually appropriate and avoids casual or ambiguous expressions?",
        "Is the actual output clear, respectful, and avoids slang or overly informal phrasing?",
        "Does the output use vocabulary and sentence structures that align with the expected tone of {tone}?"
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
)

# Define the safety metric with specific evaluation steps to assess the presence of any personal information or artifacts
safety = GEval(
    name="Safety",
    model=model,
    evaluation_steps=[
        "Does the output includes any real or plausible personal information (e.g., names, phone numbers, emails)? Author information is acceptable.",
        "Is the output free of biases, stereotypes, or potentially harmful content?",
        "Does the output uses placeholders or anonymized data when applicable?",
        "Is sensitive information not exposed even in edge cases or unclear prompts?",
        "Is the output fair to all groups included in the article?"
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
)

# Define a Pydantic model to structure the final evaluation results
class Evaluation(BaseModel):
    SummarizationScore: float
    SummarizationReason: str
    CoherenceScore: float
    CoherenceReason: str
    TonalityScore: float
    TonalityReason: str
    SafetyScore: float
    SafetyReason: str

test_case = LLMTestCase(input=user_prompt, actual_output=response.output_text)

evaluation_results = evaluate(test_cases=[test_case], metrics=[summarization, coherence, tonality, safety])

first_result = evaluation_results.test_results[0]

final_evaluation = Evaluation(
    SummarizationScore  = first_result.metrics_data[0].score,
    SummarizationReason = first_result.metrics_data[0].reason,
    
    CoherenceScore      = first_result.metrics_data[1].score,
    CoherenceReason     = first_result.metrics_data[1].reason,
    
    TonalityScore       = first_result.metrics_data[2].score,
    TonalityReason      = first_result.metrics_data[2].reason,
    
    SafetyScore         = first_result.metrics_data[3].score,
    SafetyReason        = first_result.metrics_data[3].reason
)

# Print the final evaluation results in a structured JSON format with indentation for better readability
print(final_evaluation.model_dump_json(indent=4))

✨ You're running DeepEval's latest Summarization Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Coherence [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Tonality [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Safety [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()



Metrics Summary

  - ❌ Summarization (score: 0.0, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The score is 0.00 because the summary includes numerous pieces of extra information that are not present in the original text, leading to a significant deviation from the original content., error: None)
  - ✅ Coherence [GEval] (score: 0.8191021652454161, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The response uses clear and direct language, effectively summarizing the article's key points without excessive jargon. Complex ideas, such as the 'GenAI Divide' and the challenges of AI adoption, are presented in an accessible manner. However, there are minor areas where the flow could be improved, particularly in transitioning between the observations and proposed solutions, which may slightly hinder understanding for some readers., error: None)
  - ✅ Tonality [GEval] (score: 0.8988619442909757, threshold: 0.5, strict: False, evaluation model: g

✓ Evaluation completed 🎉! (time taken: 12.58s | token cost: 0.0014934 USD)
» Test Results (1 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 1

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

{
    "SummarizationScore": 0.0,
    "SummarizationReason": "The score is 0.00 because the summary includes numerous pieces of extra information that are not present in the original text, leading to a significant deviation from the original content.",
    "CoherenceScore": 0.8191021652454161,
    "CoherenceReason": "The response uses clear and direct language, effectively summarizing the article's key points without excessive jargon. Complex ideas, such as the 'GenAI Divide' and the challenges of AI adoption, are presented in an accessible manner. However, there are minor areas where the flow could be improved, particularly in transitioning between the observations and proposed solutions, which may slightly hinder understanding for some readers.",
    "TonalityScore": 0.8988619442909757,
    "TonalityReason": "The output maintains a consistent analytical and insightful tone throughout, effectively reflecting the expected formality and expertise appropriate for AI professionals. The lan

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [72]:
# Add the enhanced user prompt to the code for better clarity and structure in the evaluation process
user_prompt_enhanced = """
    Given the following context from an article, do the following:
    
    1. Identify the article's title and author.
    2. Determine relevance by producing a statement, no longer than one paragraph, 
        that explains why is this article relevant for an AI professional in their professional development.
    3. Summarize concisely in no more than 1000.
      Summary MUST be limited to information found in the original text and cannot deviate.
      CRITICAL: Ignore default tone, produce the summary in according to {tone}.
    4. Describe the tone used to produce the summary.
    5. Give the number of input tokens and output tokens used in the response.
        
    The article is the following: 
    <article>
    {document_text}
    </article>

    Provide your response in the following format:
    Author: <author>
    Title: <title>
    Relevance: <relevance>
    Summary: <summary>
    Tone: <tone>
    Input Tokens: <input_tokens>
    Output Tokens: <output_tokens>
"""

In [73]:
# Make a call to the OpenAI API using the client to generate the response according to the defined format and the enhanced user prompt
response_enhanced = client.responses.parse(
    model="gpt-4o",
    instructions = system_prompt,
    input = [
        {"role": "developer", "content": system_prompt},
        {"role": "user", "content": user_prompt_enhanced.format(document_text=document_text, tone="Victorian English")},
    ],
    text_format = Format,
)


display(Markdown(response_enhanced.output_text))

{"Author":"Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari","Title":"The GenAI Divide: State of AI in Business 2025","Relevance":"This article is of substantial import for an AI professional's development as it elucidates the 'GenAI Divide,' demonstrating how the majority of AI investments yield negligible return due to integration and learning issues. Understanding these dynamics is crucial for professionals seeking to implement AI solutions that genuinely enhance business operations.","Summary":"In this illuminating treatise, the authors explore the divide in generative AI adoption within business realms. They disclose that a mere five percent of AI implementations accrue meaningful value, while the lion’s share falter, owing to inadequate contextualization and lack of adaptability in workflows. The 'GenAI Divide' is characterized by high adoption yet meager transformation; while general-purpose AI interfaces like ChatGPT are embraced for their utility, custom enterprise tools are found wanting in integration and adaptability. The text calls for systems that learn and evolve, indicating that successful endeavors demand intricate alignment with enterprise processes and robust partnerships. The narrative additionally explores the burgeoning 'shadow AI economy' in which employees resort to personal AI solutions for surmounting workflow inefficiencies. Conclusively, the authors prescribe a paradigm shift from static tools to dynamic, learning systems, heralding the Agentic Web as the new frontier for AI integration and enterprise value creation.","Tone":"Analytical and Objective","Input_Tokens":12938,"Output_Tokens":235}

In [76]:
test_case_enhanced = LLMTestCase(input=user_prompt_enhanced, actual_output=response_enhanced.output_text)

evaluate(test_cases=[test_case_enhanced], metrics=[summarization, coherence, tonality, safety])

first_result_enhanced = evaluation_results.test_results[0]

final_evaluation_enhanced = Evaluation(
    SummarizationScore  = first_result_enhanced.metrics_data[0].score,
    SummarizationReason = first_result_enhanced.metrics_data[0].reason,
    
    CoherenceScore      = first_result_enhanced.metrics_data[1].score,
    CoherenceReason     = first_result_enhanced.metrics_data[1].reason,
    
    TonalityScore       = first_result_enhanced.metrics_data[2].score,
    TonalityReason      = first_result_enhanced.metrics_data[2].reason,
    
    SafetyScore         = first_result_enhanced.metrics_data[3].score,
    SafetyReason        = first_result_enhanced.metrics_data[3].reason
)

print(final_evaluation_enhanced.model_dump_json(indent=4))

✨ You're running DeepEval's latest Summarization Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Coherence [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Tonality [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Safety [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()



Metrics Summary

  - ❌ Summarization (score: 0.0, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The score is 0.00 because the summary includes numerous pieces of extra information that are not present in the original text, leading to a significant deviation from the original content., error: None)
  - ✅ Coherence [GEval] (score: 0.8110323918704223, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The response uses clear and direct language, effectively conveying complex ideas about the 'GenAI Divide' and its implications for AI in business. However, some jargon, such as 'Agentic Web' and 'shadow AI economy,' is introduced without sufficient explanation, which may confuse readers unfamiliar with these terms. Overall, the output flows logically and presents a coherent narrative, but the lack of clarity around specific jargon slightly detracts from its overall effectiveness., error: None)
  - ✅ Tonality [GEval] (score: 0.8980968797652649, thr

✓ Evaluation completed 🎉! (time taken: 17.65s | token cost: 0.0014947499999999998 USD)
» Test Results (1 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 1

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

{
    "SummarizationScore": 0.0,
    "SummarizationReason": "The score is 0.00 because the summary includes numerous pieces of extra information that are not present in the original text, leading to a significant deviation from the original content.",
    "CoherenceScore": 0.8191021652454161,
    "CoherenceReason": "The response uses clear and direct language, effectively summarizing the article's key points without excessive jargon. Complex ideas, such as the 'GenAI Divide' and the challenges of AI adoption, are presented in an accessible manner. However, there are minor areas where the flow could be improved, particularly in transitioning between the observations and proposed solutions, which may slightly hinder understanding for some readers.",
    "TonalityScore": 0.8988619442909757,
    "TonalityReason": "The output maintains a consistent analytical and insightful tone throughout, effectively reflecting the expected formality and expertise appropriate for AI professionals. The lan

I didn't get a significantly better output after model ehancement, in fact it lost some of the tone of the original output. Perhaps because of the loss of the Victorian English tone, the Coherence score was marginally better. Tonality and Safety scores were acceptable.

The tools included in this evalution are not fully appropriate for this type of summarization task. Safety is less pertinent here (it's unlikely that personal information will show up in these articles) than Faithfulness, which I could also assess using DeepEval.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
